# Agente de revisión de documentos y planos con Gemini
Usa la API de Gemini (plan gratuito de Google AI Studio o de pago): lee planos con visión y tiene contexto largo, sin necesitar GPU.

Antes de ejecutar:
1. Crea una clave en https://aistudio.google.com/apikey.
2. En Kaggle: *Add-ons → Secrets* → agrega `GEMINI_API_KEY`, y activa *Settings → Internet: On*. No necesitas GPU.
3. Sube tus archivos como Dataset y los `.py` y checklists del proyecto (`doc_review_agent.py`, `plan_review.py`, `local_llm.py`, `gemini_llm.py`, `doc_chat.py`, `checklist_default.md`, `checklist_planos.md`).

⚠️ **Confidencialidad:** en el plan gratuito Google puede usar lo que envías para mejorar sus productos y revisores humanos pueden leerlo. No subas planos o documentos confidenciales sin activar facturación (con facturación ese uso se excluye) o sin autorización.
⚠️ Los límites gratuitos son bajos y cambian: el agente espera y reintenta ante errores 429.

In [ ]:
# Los avisos rojos 'dependency resolver ... google-colab ...' son normales en Kaggle y no impiden continuar.
!pip install -q -U google-genai pypdf python-docx openpyxl pymupdf pillow

In [ ]:
# Código del agente: se descarga/actualiza desde GitHub (siempre la última versión).
# Solo si no hay Internet se usan los .py que estén en /kaggle/input o /kaggle/working.
import glob, os, subprocess, sys
CODE_DIR = "/kaggle/working/agente"
if os.path.isdir(os.path.join(CODE_DIR, ".git")):
    r = subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only", "-q"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/gallant-fermat-x83ohv",
                        "https://github.com/mrubilar84-ship-it/PRUEBA-AGENTE_1", CODE_DIR], capture_output=True, text=True)
if r.returncode:
    hits = glob.glob("/kaggle/input/**/doc_review_agent.py", recursive=True)
    if not hits:
        raise RuntimeError("No pude descargar el código (¿Internet: On?) ni hay copia en /kaggle/input:\n" + r.stderr)
    CODE_DIR = os.path.dirname(hits[0])
    print("Sin conexión a GitHub: uso la copia de", CODE_DIR)
sys.path.insert(0, CODE_DIR)
print("Código en:", CODE_DIR)

from doc_review_agent import ReviewConfig, review_folder
import os
from kaggle_secrets import UserSecretsClient
os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
from gemini_llm import GeminiLLM

In [ ]:
# Modelo por defecto: gemini-3.8-flash. Verifica en AI Studio qué modelos tienes y su cuota;
# para más calidad puedes usar uno "pro" (más lento y con cuota gratuita menor): GeminiLLM("<modelo>").
llm = GeminiLLM()
print(llm.check())          # prueba rápida: debe imprimir "listo". Si falla, el mensaje dice por qué.
# print(llm.list_models())  # modelos disponibles para tu clave (si el modelo por defecto no existe)

In [ ]:
cfg = ReviewConfig(
    plan_tiles=True,   # False = más rápido (solo vista general + cajetín)
    extra_instructions="Proyecto: <nombre>. Normas aplicables: <ej. AISC 360-16, NCh433>. Unidades: SI.",
)
IN_DIR = "/kaggle/input/documentos-ingenieria"   # nombre o URL de tu dataset (o una carpeta de /kaggle/input)
resultado = review_folder(IN_DIR, "/kaggle/working/informe", cfg, llm=llm)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/informe/informe.md", encoding="utf-8").read()))

## Chat con los documentos
Haz preguntas, pide resúmenes o consulta los hallazgos de la revisión. Responde solo con lo que encuentra en los documentos y cita archivo y página.

## Interpretación de planos
Extrae de cada hoja qué muestra: tipo de plano, cajetín, elementos, cotas, materiales, notas y referencias. Se guarda en `planos_interpretados.md/json` y alimenta el chat (si tu carpeta no tiene planos, omite esta celda y la línea `add_plans`).

In [ ]:
from plan_review import interpret_folder
planos = interpret_folder(IN_DIR, "/kaggle/working/informe", cfg, llm=llm)
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/informe/planos_interpretados.md", encoding="utf-8").read()))

In [ ]:
from doc_chat import DocChat
chat = DocChat(llm)
chat.add_folder(IN_DIR)
chat.add_review("/kaggle/working/informe/revision.json")   # opcional: permite preguntar por los hallazgos
chat.add_plans(planos)   # planos interpretados: el chat los usa y mira la hoja relevante

In [ ]:
print(chat.ask("¿Cuáles son los hallazgos críticos y mayores?"))
print(chat.ask("¿Qué norma se aplica y qué factores de seguridad se usan?"))
print(chat.ask("¿Qué muestra el plano de la hoja 1 y cuáles son sus cotas principales?"))
print(chat.ask("¿Qué materiales y normas se indican en los planos?"))

In [ ]:
print(chat.summarize())            # todos los documentos; o chat.summarize("archivo.pdf")

In [ ]:
# Preguntas sobre lo que SE VE en una hoja de plano (PDF y número de hoja)
print(chat.ask_image("¿Las cotas parciales suman la cota total? ¿Qué datos faltan en el cajetín?",
                     "/kaggle/input/documentos-ingenieria/plano.pdf", page=1))

In [ ]:
chat.repl()   # chat interactivo: escribe tus preguntas; /resumen, /reset, /salir